# 05 - Retrain Ultra-CAN on TDL-D (CONV1D)

Trains and evaluates one Ultra-CAN variant on the 3GPP TDL-D channel (300 ns RMS delay spread, 25% clutter) and then regenerates the retraining-control figure for that variant.

**Before running:** upload the repository, with its `results/` directory, to `/content/ultra-can`. Results are written **live** to `/content/drive/MyDrive/ultra_can/retrain_tdl_d/`, so a disconnect only loses the current epoch checkpoint, never the data already written.

The notebook refuses to train if the resolved channel is not `tdl_3gpp`.

## 0. Configuration

In [ ]:
ARCH = "conv1d"
RUN_NAME = "retrain_tdl_d"
MODE = "full"          # use "fast" for a pipeline smoke test
REPO = "/content/ultra-can"
DRIVE_ROOT = "/content/drive/MyDrive/ultra_can"
OUT_ROOT = f"{DRIVE_ROOT}/{RUN_NAME}"


## 1. Mount Drive (results are written there)

In [ ]:
import os

DRIVE_OK = True
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception as exc:  # running outside Colab
    DRIVE_OK = False
    print("Drive not mounted:", exc)
    DRIVE_ROOT = os.path.abspath("./drive_sim")
    OUT_ROOT = os.path.join(DRIVE_ROOT, RUN_NAME)
os.makedirs(OUT_ROOT, exist_ok=True)
print("output root:", OUT_ROOT, "| drive:", DRIVE_OK)


## 2. Resolve the repository

In [ ]:
import os

if not os.path.isdir(os.path.join(REPO, "src")):
    raise SystemExit(
        "Repository not found at /content/ultra-can. Upload the full repo "
        "(including results/) before running this notebook."
    )
os.chdir(REPO)
print("repo:", os.getcwd())
print("results present:", os.path.isdir("results/full"))


## 3. Dependencies and GPU

In [ ]:
!python -m pip install -q -r requirements.txt
import platform, tensorflow as tf
print(platform.platform())
print("TensorFlow", tf.__version__, "| GPU", tf.config.list_physical_devices("GPU"))


## 4. Resolve the TDL-D channel (must print `tdl_3gpp`)

In [ ]:
import subprocess, sys

command = [
    sys.executable, "src/experiments/runner.py",
    "--experiments", "ber_vs_snr", "--mode", MODE,
    "--scenario", "k3_b_tdl_d_retrained",
    "--channel", "b_tdl_d", "--model", ARCH,
    "--output-dir", OUT_ROOT, "--dry-run",
]
plan = subprocess.run(command, capture_output=True, text=True)
print(plan.stdout[-4000:])
print(plan.stderr[-2000:])
assert plan.returncode == 0, "dry-run failed"
assert "tdl_3gpp" in plan.stdout + plan.stderr, (
    "the channel did not resolve to tdl_3gpp: check --channel b_tdl_d"
)
print("channel check: OK (tdl_3gpp)")


## 5. Train and evaluate

In [ ]:
import subprocess, sys

command = [
    sys.executable, "src/experiments/runner.py",
    "--experiments", "ber_vs_snr", "--mode", MODE,
    "--scenario", "k3_b_tdl_d_retrained",
    "--channel", "b_tdl_d", "--model", ARCH,
    "--output-dir", OUT_ROOT, "--force",
]
print(" ".join(command), flush=True)
subprocess.run(command, check=True)


## 6. Validate the run and merge it into the repo results

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import yaml

run = Path(OUT_ROOT) / MODE / "ber_vs_snr" / "k3_b_tdl_d_retrained" / ARCH
metrics_path = run / "metrics.csv"
assert metrics_path.is_file(), f"missing {metrics_path}"
frame = pd.read_csv(metrics_path)
print(frame[["snr_db", "ber", "corr_tau", "exact_tau"]].to_string(index=False))
print("pooled BER >= 5 dB:", float(frame[frame.snr_db >= 5].ber.mean()))

cfg_path = Path(OUT_ROOT) / MODE / "ber_vs_snr" / "logs" / "config_used.yaml"
if not cfg_path.is_file():
    cfg_path = run / "logs" / "config_used.yaml"
cfg = yaml.safe_load(cfg_path.read_text())
print("channel model:", cfg["channel"]["model"], "| hold:", cfg["channel"]["hold_mode"])
assert cfg["channel"]["model"] == "tdl_3gpp", "this run is NOT on TDL-D"

dst = Path(REPO) / "results" / "full" / "ber_vs_snr" / "k3_b_tdl_d_retrained" / ARCH
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run, dst, dirs_exist_ok=True)
print("copied to", dst)


## 7. Regenerate the retraining figure for this variant

In [ ]:
import os, subprocess, sys

os.environ["RETRAIN_ARCHS"] = ARCH
subprocess.run([sys.executable, "scripts/figures/make_fig_retraining_control.py"],
               check=True)
print("figure:", os.path.abspath("figures/retraining_control.pdf"))


## 8. Archive on Drive

In [ ]:
import shutil
from pathlib import Path

run = Path(OUT_ROOT) / MODE / "ber_vs_snr" / "k3_b_tdl_d_retrained" / ARCH
archive = Path(DRIVE_ROOT) / f"retrain_tdl_d_{ARCH}"
for old in (archive.with_suffix(".zip"),):
    if old.exists():
        old.unlink()
shutil.make_archive(str(archive), "zip", root_dir=run.parent, base_dir=run.name)
print("archive:", archive.with_suffix(".zip"))

fig = Path("figures/retraining_control.pdf")
if fig.is_file():
    shutil.copy2(fig, Path(DRIVE_ROOT) / f"retraining_{ARCH}.pdf")
    print("figure copied to Drive")


## Outputs

- `{OUT_ROOT}/full/ber_vs_snr/k3_b_tdl_d_retrained/conv1d/` (checkpoint, `metrics.csv`, logs)
- `{DRIVE_ROOT}/retrain_tdl_d_conv1d.zip`
- `{DRIVE_ROOT}/retraining_conv1d.pdf`

Send back the zip (or just the `metrics.csv` and the `logs/` directory). The final figure is regenerated after merging both variants.